# CSIS3764 Exam Q2 — End-of-Year 2023
## Patients Dataset — ML Classifiers

## 2.1 — Import dataset

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

patients = pd.read_csv('patients.csv')
print(patients.head())

## 2.2 — Brief summary

In [ ]:
# Records 50 to 60 (inclusive)
print(patients.iloc[49:61])

# Number of records and features
print('Records, Features:', patients.shape)

# Unique values per feature
print(patients.nunique())

# Missing values
print(patients.isnull().sum())

## 2.3 — Probability: male, A+ blood, Diabetes

In [ ]:
male_apos = patients[
    (patients['Gender'] == 'Male') &
    (patients['Blood Type'] == 'A+')
]
diabetes_count = male_apos[male_apos['Medical Condition'] == 'Diabetes'].shape[0]
probability = (diabetes_count / male_apos.shape[0]) * 100
print(f'Probability: {probability:.1f}%')

## 2.4 — Rarest blood type

In [ ]:
blood_counts = patients['Blood Type'].value_counts()
print(blood_counts)
print(f"Rarest blood type: {blood_counts.idxmin()} ({blood_counts.min()} patients)")

## 2.5 — Patient with largest bill

In [ ]:
max_bill_idx = patients['Billing Amount'].idxmax()
print(patients.loc[max_bill_idx, ['Name', 'Billing Amount']])

## 2.6 — Dates with most patients admitted

In [ ]:
patients['Date of Admission'] = pd.to_datetime(patients['Date of Admission'])
admissions = patients.groupby('Date of Admission').size()
max_count = admissions.max()
print('Most admissions:', max_count)
print(admissions[admissions == max_count])

## 2.7 — Statistical summary

In [ ]:
print(patients.describe(include='all'))

In [ ]:
print("""
2.7.1 Inference regarding blood type:
From the statistical summary, the blood type feature shows multiple
unique values with relatively even distribution across the dataset.
The most frequent blood type can be identified from the top/freq fields.
This suggests no single blood type dominates the patient population,
indicating a diverse sample. Any blood type with significantly fewer
records may indicate underrepresentation in the dataset.
""")

## 2.8 — Correlation matrix + heatmap

In [ ]:
numeric_df = patients.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()
print(corr_matrix)

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
print("""
2.8.1 Inference from correlation matrix:
Feature pairs with higher absolute correlation values (closer to 1 or -1)
indicate a stronger linear relationship. These highly correlated features
may cause multicollinearity in the model, and one of each pair could
potentially be dropped without losing significant information. Features
with correlation close to 0 have little linear relationship with each other.
""")

## 2.9 — Test result counts

In [ ]:
print(patients['Test Results'].value_counts())

## 2.10 — Pie chart of test results

In [ ]:
counts = patients['Test Results'].value_counts()
plt.figure(figsize=(6, 6))
plt.pie(counts, labels=counts.index,
        autopct=lambda p: f'{p:.0f}%',
        startangle=90)
plt.title('Test Result Distribution')
plt.show()

In [ ]:
print("""
2.11 Recommendation on balancing:
If the pie chart shows a roughly equal split between test result classes,
the dataset is already balanced and no further resampling is needed.
If one class significantly dominates (e.g. 60%+ of records), the data
should be balanced using oversampling (SMOTE) or undersampling to prevent
the model from being biased toward the majority class and ignoring
minority classes, which would result in misleading accuracy scores.
""")

## 2.12 — Remove features with more than 10 unique text values

In [ ]:
text_columns = patients.select_dtypes(include=['object']).columns.tolist()
for col in text_columns:
    if patients[col].nunique() > 10:
        patients = patients.drop(columns=[col])
        print(f"Dropped '{col}'")
print('Remaining columns:', patients.columns.tolist())

## 2.13 — Remove insurance provider, billing amount, room number

In [ ]:
cols_to_drop = ['Insurance Provider', 'Billing Amount', 'Room Number']
existing = [c for c in cols_to_drop if c in patients.columns]
patients = patients.drop(columns=existing)
print('Dropped:', existing)
print('Remaining:', patients.columns.tolist())

## 2.14 — Convert text to numeric

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode target
le = LabelEncoder()
patients['Test Results'] = le.fit_transform(patients['Test Results'])
print('Test Results classes:', le.classes_)

# Encode input features
text_cols = patients.select_dtypes(include=['object']).columns.tolist()
for col in text_cols:
    unique_vals = patients[col].nunique()
    if unique_vals == 2:
        vals = patients[col].unique()
        patients[col] = patients[col].map({vals[0]: 0, vals[1]: 1})
        print(f"Binary encoded: '{col}'")
    elif unique_vals > 2:
        dummies = pd.get_dummies(patients[col], prefix=col)
        patients = pd.concat([patients.drop(columns=[col]), dummies], axis=1)
        print(f"One-hot encoded: '{col}'")

patients = patients.astype({col: int for col in patients.select_dtypes('bool').columns})
print('Remaining object cols:', patients.select_dtypes(include=['object']).columns.tolist())
print(patients.head())

## 2.15 — X, y split + train/test split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

patients = patients.select_dtypes(exclude=['datetime64[ns]'])

X = patients.drop(columns=['Test Results'])
y = patients['Test Results']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)
print('X_test: ', X_test.shape)
print('y_test: ', y_test.shape)

## 2.16 — Train classifiers + learning curves (k=10)

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import KFold, cross_val_score, learning_curve

models = [
    ('NB',  GaussianNB()),
    ('LR',  LogisticRegression(solver='lbfgs', max_iter=1000)),
    ('SVM', SVC(gamma='auto'))
]

kfold = KFold(n_splits=10, shuffle=True, random_state=42)
results_acc = []
results_f1  = []
names       = []

def plot_learning_curve(model, name, X, y, cv):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=cv, scoring='accuracy',
        train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1)
    train_mean = train_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    val_mean   = val_scores.mean(axis=1)
    val_std    = val_scores.std(axis=1)
    plt.figure(figsize=(8, 5))
    plt.plot(train_sizes, train_mean, label='Training score', color='blue')
    plt.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.1, color='blue')
    plt.plot(train_sizes, val_mean, label='CV score', color='orange')
    plt.fill_between(train_sizes, val_mean-val_std, val_mean+val_std, alpha=0.1, color='orange')
    plt.title(f'Learning Curve — {name}')
    plt.xlabel('Training Size')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.tight_layout()
    plt.show()

for name, model in models:
    acc_scores = cross_val_score(model, X_train_scaled, y_train, cv=kfold, scoring='accuracy')
    f1_scores  = cross_val_score(model, X_train_scaled, y_train, cv=kfold, scoring='f1_weighted')
    results_acc.append(acc_scores)
    results_f1.append(f1_scores)
    names.append(name)
    print(f'{name} (acc): {acc_scores.mean():.4f}')
    print(f'{name} (f1):  {f1_scores.mean():.4f}')
    print(f'{name} done\n')
    plot_learning_curve(model, name, X_train_scaled, y_train, kfold)

## 2.17 — Best model predictions

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

f1_means   = [s.mean() for s in results_f1]
best_index = f1_means.index(max(f1_means))
best_name  = names[best_index]
best_clf   = dict(models)[best_name]
print(f'Best model: {best_name} (F1 = {f1_means[best_index]:.4f})')

best_clf.fit(X_train_scaled, y_train)
y_pred = best_clf.predict(X_test_scaled)

print(f'Test Accuracy: {accuracy_score(y_test, y_pred):.4f}')

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix — {best_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

print('Classification Report:')
print(classification_report(y_test, y_pred))

## 2.18 — Inference and recommendations

In [ ]:
print(f"""
=== Inference from Learning Curves, Metrics and Scores ===

1. Learning Curves:
   - If the training score is much higher than the CV score, the model
     is overfitting. Adding more data or regularisation may help.
   - If both scores are low and converge together, the model is underfitting.
     A more complex model or better features may help.
   - Converging curves at a high value indicate a good fit.

2. Cross-Validation Scores:
   - A high CV accuracy and F1 score indicate the model generalises well.
   - A large standard deviation across folds suggests instability.

3. Confusion Matrix:
   - Off-diagonal values show misclassifications between classes.
   - In a medical context, false negatives are especially concerning
     as missed diagnoses carry significant risk.

4. Classification Report:
   - Precision, recall and F1 per class reveal which test result
     categories the model struggles to predict correctly.

=== Recommendations ===
   - If accuracy is near 33% (3-class random baseline), the features
     lack sufficient predictive signal — consider feature engineering.
   - Apply StandardScaler if not already done — SVM and LR are
     sensitive to unscaled features.
   - Consider ensemble methods like Random Forest or XGBoost for
     better non-linear classification.
   - If class imbalance exists, apply SMOTE or class_weight='balanced'.
""")